In [3]:
!pip install kagglehub
import kagglehub
path = kagglehub.dataset_download("gpiosenka/cards-image-datasetclassification")
print(path)

100%|██████████| 385M/385M [00:02<00:00, 138MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/gpiosenka/cards-image-datasetclassification/versions/2


In [4]:
import os

DATA_DIR = "/root/.cache/kagglehub/datasets/gpiosenka/cards-image-datasetclassification/versions/2"

for item in os.listdir(DATA_DIR):
    print(item)

train
14card types-14-(200 X 200)-94.61.h5
valid
test
cards.csv
53cards-53-(200 X 200)-100.00.h5


In [5]:
train_path = os.path.join(DATA_DIR, "train")
print(os.listdir(train_path)[:10])   # first 10 classes/folders
print("Total classes:", len(os.listdir(train_path)))

['three of spades', 'three of clubs', 'seven of diamonds', 'king of spades', 'six of clubs', 'jack of spades', 'seven of hearts', 'ace of spades', 'ace of clubs', 'queen of diamonds']
Total classes: 53


In [6]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

IMG_SIZE = 200

ssl_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, "train"), transform=ssl_transform)
test_dataset  = datasets.ImageFolder(root=os.path.join(DATA_DIR, "test"), transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

print("Total train images:", len(train_dataset))
print("Total test images:", len(test_dataset))

Total train images: 7624
Total test images: 265


In [7]:
from torch.utils.data import Dataset
from PIL import Image

class SimCLRDataset(Dataset):
    def __init__(self, image_folder_dataset, transform):
        self.dataset = image_folder_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        path, _ = self.dataset.samples[idx]
        img = Image.open(path).convert("RGB")
        view1 = self.transform(img)
        view2 = self.transform(img)   # same image, 2 different augmentations
        return view1, view2

# Wrap our existing train_dataset (ImageFolder) with SimCLR logic
simclr_train = SimCLRDataset(train_dataset, ssl_transform)
simclr_loader = DataLoader(simclr_train, batch_size=128, shuffle=True, num_workers=2, drop_last=True)

print("SimCLR batches per epoch:", len(simclr_loader))

SimCLR batches per epoch: 59


In [8]:
import torch.nn as nn
import torchvision.models as models

class SimCLRModel(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()
        backbone = models.resnet18(weights=None)   # scratch training, no pretrained
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])  # remove final FC
        self.feature_dim = backbone.fc.in_features  # 512

        self.projection_head = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim),
            nn.ReLU(),
            nn.Linear(self.feature_dim, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x).squeeze()   # feature representation
        z = self.projection_head(h)     # projected embedding for contrastive loss
        return h, z

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimCLRModel().to(device)
print(model)
print("Device:", device)

SimCLRModel(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=T

In [9]:
import torch
import torch.nn as nn
import torchvision.models as models

class SimCLRModel(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()
        backbone = models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])
        self.feature_dim = backbone.fc.in_features

        self.projection_head = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim),
            nn.ReLU(),
            nn.Linear(self.feature_dim, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x).squeeze()
        z = self.projection_head(h)
        return h, z

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimCLRModel().to(device)
print(model)
print("Device:", device)

SimCLRModel(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=T

In [13]:
import torch.nn.functional as F

def nt_xent_loss(z1, z2, temperature=0.5):
    batch_size = z1.shape[0]
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    representations = torch.cat([z1, z2], dim=0)  # (2N, dim)
    similarity_matrix = torch.matmul(representations, representations.T)

    # Mask self-similarity
    mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z1.device)
    similarity_matrix = similarity_matrix / temperature

    # Positive pairs: (i, i+N) and (i+N, i)
    positives = torch.cat([
        torch.diag(similarity_matrix, batch_size),
        torch.diag(similarity_matrix, -batch_size)
    ], dim=0)

    similarity_matrix.masked_fill_(mask, -9e15)  # remove self-similarity
    negatives = similarity_matrix

    logits = torch.cat([positives.unsqueeze(1), negatives], dim=1)
    labels = torch.zeros(2 * batch_size, dtype=torch.long).to(z1.device)

    loss = F.cross_entropy(logits, labels)
    return loss

In [14]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=3e-4)
num_epochs = 20   # start with 20, increase pannalam later

model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for view1, view2 in simclr_loader:
        view1, view2 = view1.to(device), view2.to(device)

        _, z1 = model(view1)
        _, z2 = model(view2)

        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(simclr_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.4f}")

Epoch [1/20] - Loss: 4.3351
Epoch [2/20] - Loss: 4.0135
Epoch [3/20] - Loss: 3.9335
Epoch [4/20] - Loss: 3.8915
Epoch [5/20] - Loss: 3.8597
Epoch [6/20] - Loss: 3.8398
Epoch [7/20] - Loss: 3.8206
Epoch [8/20] - Loss: 3.8110
Epoch [9/20] - Loss: 3.7960
Epoch [10/20] - Loss: 3.7845
Epoch [11/20] - Loss: 3.7751
Epoch [12/20] - Loss: 3.7732
Epoch [13/20] - Loss: 3.7642
Epoch [14/20] - Loss: 3.7611
Epoch [15/20] - Loss: 3.7513
Epoch [16/20] - Loss: 3.7498
Epoch [17/20] - Loss: 3.7471
Epoch [18/20] - Loss: 3.7404
Epoch [19/20] - Loss: 3.7384
Epoch [20/20] - Loss: 3.7341


In [15]:
torch.save(model.encoder.state_dict(), "self_supervised_encoder.pth")
print("Encoder saved!")

Encoder saved!


In [18]:
import torch.nn as nn

class LinearClassifier(nn.Module):
    def __init__(self, encoder, num_classes, feature_dim=512):
        super().__init__()
        self.encoder = encoder
        for param in self.encoder.parameters():
            param.requires_grad = False   # freeze encoder - only classifier learns
        self.fc = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            features = self.encoder(x).squeeze()
        return self.fc(features)

num_classes = len(train_dataset.classes)  # 53
classifier_model = LinearClassifier(model.encoder, num_classes).to(device)

# Only train the classifier head
optimizer_cls = optim.Adam(classifier_model.fc.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Use eval_transform loader for actual labels (train_dataset already has labels via ImageFolder)
eval_train_loader = DataLoader(
    datasets.ImageFolder(root=os.path.join(DATA_DIR, "train"), transform=eval_transform),
    batch_size=64, shuffle=True, num_workers=2
)

In [19]:
num_epochs_cls = 10
classifier_model.train()
for epoch in range(num_epochs_cls):
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in eval_train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = classifier_model(images)
        loss = criterion(outputs, labels)

        optimizer_cls.zero_grad()
        loss.backward()
        optimizer_cls.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs_cls}] - Loss: {total_loss/len(eval_train_loader):.4f} - Train Acc: {acc:.2f}%")

Epoch [1/10] - Loss: 2.8343 - Train Acc: 25.79%
Epoch [2/10] - Loss: 2.4288 - Train Acc: 34.60%
Epoch [3/10] - Loss: 2.3424 - Train Acc: 36.44%
Epoch [4/10] - Loss: 2.3022 - Train Acc: 36.80%
Epoch [5/10] - Loss: 2.2631 - Train Acc: 38.14%
Epoch [6/10] - Loss: 2.2460 - Train Acc: 38.96%
Epoch [7/10] - Loss: 2.2046 - Train Acc: 39.99%
Epoch [8/10] - Loss: 2.1768 - Train Acc: 40.27%
Epoch [9/10] - Loss: 2.1550 - Train Acc: 40.79%
Epoch [10/10] - Loss: 2.1432 - Train Acc: 41.53%


In [20]:
classifier_model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = classifier_model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

test_acc = 100 * correct / total
print(f"Test Accuracy: {test_acc:.2f}%")

Test Accuracy: 34.72%


In [21]:
torch.save(classifier_model.state_dict(), "full_classifier_model.pth")
print("Full classifier model saved!")

Full classifier model saved!


In [25]:
readme_content = """
## Results

- Self-Supervised Pretraining (SimCLR): 20 epochs, Loss: 4.33 -> 3.73
- Linear Evaluation (Classifier on frozen encoder):
  - Train Accuracy: 41.53%
  - Test Accuracy: 34.72%
- Dataset: 53-class playing card dataset (7624 train / 265 test images)
- Backbone: ResNet18 (trained from scratch, no pretrained weights)
"""

with open("README_results.md", "w") as f:
    f.write(readme_content)

print("README section saved!")

README section saved!


In [27]:
from google.colab import files
files.download('self_supervised_encoder.pth')
files.download('full_classifier_model.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
from google.colab import files
files.download('README_results.md')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>